# Digital Payment Transactions Reversal Prediction
Logistic Regression Project - predicting `is_reversed`

## Step 1: Import Libraries and Load the Data

This cell just brings in all the tools (libraries) I need for the whole project.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              roc_auc_score, roc_curve)

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

This cell reads my CSV file and loads it into a table called `df`.

In [3]:
df = pd.read_csv("fintech_dataset_5000_cleaned_is_reversed.csv")
df.head()

,transaction_id,timestamp,customer_id,customer_name,age,city,state,country,transaction_type,category,...,account_balance,bank_name,merchant_id,device_type,transaction_status,is_reversed,credit_score,kyc_level,risk_level,is_fraud
0,TXN0000001,01-01-2023 01:16,CUST969138,Maria Sharma,29,Dubai,Dubai,UAE,Refund,Fuel,...,17544.24,Bank of America,MER9245,ATM,Completed,0,638,Enhanced,Low,False
1,TXN0000002,01-01-2023 05:05,CUST673602,James Iyer,32,Pune,Maharashtra,India,Purchase,Clothing,...,9168.77,Kotak Mahindra,MER6118,POS,Pending,0,780,Standard,Medium,False
2,TXN0000003,01-01-2023 08:48,CUST620155,Rohan Brown,67,Hyderabad,Telangana,India,Refund,Groceries,...,333045.57,Axis Bank,MER7701,ATM,Completed,0,740,Basic,High,False
3,TXN0000004,01-01-2023 20:32,CUST630122,Pooja Brown,61,Berlin,Berlin,Germany,Transfer,Utilities,...,4406.20,Barclays,MER6845,POS,Completed,0,678,Basic,Medium,False
4,TXN0000005,02-01-2023 01:24,CUST819971,Amanda Johnson,65,Kolkata,West Bengal,India,Purchase,Clothing,...,39587.04,HSBC,MER4797,Web,Completed,0,634,Standard,Low,False


This cell tells me how many rows and columns my data has.

In [ ]:
df.shape

This cell shows the data type of every column, so I know which are numbers and which are text.

In [ ]:
df.info()

This cell gives me a quick summary (average, min, max) of the number columns.

In [ ]:
df.describe()

This cell gives the same summary but for all columns, including text ones.

In [ ]:
df.describe(include = "all")

This cell lists all the column names, so I can decide which ones to keep, drop, or use as target.

In [ ]:
df.columns

## Step 2: Understand the Target Column
My topic was originally *Digital Payment Transaction Reversal Prediction*, so my target column is **`is_reversed`**, not `is_fraud`. `is_fraud` is a different problem, so I will drop it later to avoid confusion and data leakage.

This cell checks how balanced my target column `is_reversed` is (how many 0s vs 1s).

In [ ]:
df["is_reversed"].value_counts()

In [ ]:
df["is_reversed"].value_counts(normalize=True) * 100

#### What this tells us
- Out of all transactions, only **14.1% got reversed** (705 out of 5,000) - the rest (85.9%) went through fine.
- This is called an **imbalanced dataset** - reversed transactions are the "rare" event we're trying to catch.
- Because it's imbalanced, plain accuracy can be misleading later - a model that just guesses "not reversed" every time would already be right 86% of the time without learning anything useful.


In [ ]:
df.isnull().sum()

#### What this tells us
- **Zero missing values** anywhere - the data is clean, no need to fill in gaps or drop rows for missing info.


In [ ]:
df.duplicated().sum()

#### What this tells us
- **Zero duplicate rows** - every transaction in here is unique, so no risk of double-counting.


In [ ]:
df1 = df.drop_duplicates()
df1.shape

In [ ]:
df1 = df1.drop(columns=["transaction_id", "customer_id", "customer_name", "merchant_id","is_fraud"])
df1.shape

In [ ]:
df1 = df1.drop(columns=["transaction_status"])
df1.shape

In [ ]:
df["timestamp"].dtype

In [ ]:
df.timestamp

In [ ]:
df1["timestamp"]=pd.to_datetime(df1["timestamp"] , format = "%d-%m-%Y %H:%M")

In [ ]:
df1["txn_hour"] = df1["timestamp"].dt.hour
df1["txn_day"] = df1["timestamp"].dt.day
df1["txn_month"] = df1["timestamp"].dt.month
df1[["timestamp","txn_hour","txn_day","txn_month"]].head()

In [ ]:
df1 = df1.drop(columns=["timestamp"])

In [ ]:
print("Unique cities:" , df1["city"].nunique())
print("Unique states:" , df1["state"].nunique())

In [ ]:
df1 = df1.drop(columns=["city", "state"])

In [ ]:
df1.columns

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x="is_reversed", data=df1, palette="pastel")
plt.title("Reversed vs Not Reversed Transactions")
plt.show()

#### What this tells us
- This chart makes the imbalance we saw earlier super obvious - the orange bar (reversed) is way shorter than the blue bar (not reversed).
- Visually confirms: reversals are the minority class, not reversing is the "normal" outcome.


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df1["amount"], bins=30, kde=True, color="skyblue")
plt.title("Distribution of Transaction Amount")
plt.xlabel("Amount")
plt.show()

#### What this tells us
- Most transactions are for **small amounts**, with a few huge outlier transactions stretching the chart out.
- This is a classic **right-skewed** distribution - lots of small/medium purchases, a handful of big-ticket ones.


In [ ]:
plt.figure(figsize=(7,5))
sns.boxplot(x="is_reversed", y="amount", data=df1, palette="Set2")
plt.title("Amount by Reversal Status")
plt.show()

#### What this tells us
- Reversed and "not reversed" transactions have **basically the same amount range** - the boxes look almost identical.
- This means **transaction amount alone doesn't tell us much** about whether something gets reversed.


In [ ]:
plt.figure(figsize=(7,5))
sns.boxplot(x="is_reversed", y="account_balance", data=df1, palette="Set3")
plt.title("Account Balance by Reversal Status")
plt.show()

#### What this tells us
- Same story here - account balance looks **nearly identical** for reversed vs "not reversed" transactions.
- So a customer's balance isn't a strong clue either for predicting reversals.


In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x="transaction_type", data=df1, palette="pastel")
plt.title("Number of Transactions by Type")
plt.xticks(rotation=30)
plt.show()

#### What this tells us
- Transaction types (Refund, Purchase, Transfer, etc.) are all **fairly evenly used** - no single type totally dominates the data.


In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(x="transaction_type", y="is_reversed", data=df1, estimator=np.mean, palette="viridis")
plt.title("Average Reversal Rate by Transaction Type")
plt.ylabel("Reversal Rate")
plt.xticks(rotation=30)
plt.show()

#### What this tells us
- **Deposits** have the highest reversal rate followed by EMI.
- **Withdrawals** have the lowest reversal rate.
- So transaction type does have *some* effect, but the gap between highest and lowest isn't huge - this alone won't make a super strong predictor.


In [ ]:
plt.figure(figsize=(7,5))
sns.barplot(x="device_type", y="is_reversed", data=df1, estimator=np.mean, palette="magma")
plt.title("Average Reversal Rate by Device Type")
plt.ylabel("Reversal Rate")
plt.show()

#### What this tells us
- Reversal rates across devices (ATM, POS, Web, Tablet, Mobile) are all **close to each other**, roughly 13–15%.
- **Web and Tablet** are slightly higher, **POS** slightly lower - but honestly the difference is pretty small.


In [ ]:
plt.figure(figsize=(6,5))
sns.barplot(x="risk_level", y="is_reversed", data=df1, estimator=np.mean, palette="rocket")
plt.title("Average Reversal Rate by Risk Level")
plt.ylabel("Reversal Rate")
plt.show()

#### What this tells us
- This one's surprising - **Low, Medium, and High risk levels all have very similar reversal rates** (roughly 13–15%).
- We'd expect "High Risk" transactions to reverse a lot more often, but the data doesn't really show that here.
- This hints that the `risk_level` label might not be as predictive as its name suggests.


In [ ]:
plt.figure(figsize=(9,7))
numeric_df = df1.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

#### What this tells us - this one's important!
- Look at the `is_reversed` row/column - **every correlation number is close to 0** (like 0.00, -0.01, -0.03).
- In plain English: **none of the numeric columns (age, amount, balance, credit score, hour, day, month) have a real relationship with whether a transaction gets reversed.**
- This is a big clue for later - if nothing strongly correlates with the target, we shouldn't expect the model to perform amazingly well.


In [ ]:
x = df1.drop("is_reversed",axis=1)
y = df["is_reversed"]
print(x.shape , y.shape)

In [ ]:
x.select_dtypes(include ="object").columns

In [ ]:
x = pd.get_dummies(x,columns=['country', 'transaction_type', 'category', 'currency', 'bank_name',
       'device_type', 'kyc_level', 'risk_level'],drop_first = True ,dtype="int")
x.head()

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("Train size:", x_train.shape)
print("Test size:", x_test.shape)

In [ ]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [ ]:
log_model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
log_model.fit(x_train_scaled,y_train)

In [ ]:
y_pred = log_model.predict(x_test_scaled)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

#### What this tells us
- The model got **52.9% accuracy** - only slightly better than a coin flip.
- Remember, just guessing "not reversed" every single time would give ~86% accuracy - so 52.9% actually sounds worse, but that's expected because we asked the model to *balance* classes (`class_weight="balanced"`) instead of just chasing raw accuracy.
- **Lesson**: for imbalanced problems like this, accuracy alone is a misleading scorecard.


In [ ]:
cm = confusion_matrix(y_test , y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Not Reversed", "Reversed"],
            yticklabels=["Not Reversed", "Reversed"])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

#### What this tells us
- Out of 139 actual reversed transactions, the model correctly caught **71 of them** (true positives) but missed **68**.
- It also wrongly flagged **403 normal transactions** as "reversed" (false alarms).
- In simple terms: the model is decent at *catching* reversals but throws a **lot of false alarms** along the way.


In [ ]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

#### What this tells us
- **Precision = 15%** -> when the model says "this will be reversed," it's only right 15% of the time. Lots of false alarms.
- **Recall = 51%** -> out of all the real reversed transactions, it catches about half.
- **F1 Score = 23%** -> the overall balance between precision and recall is low, meaning the model still has a long way to go.


In [ ]:
print(classification_report(y_test, y_pred))

#### What this tells us
- For **normal transactions (class 0)**, the model does decent precision (87%) but only catches half of them (53% recall).
- For **reversed transactions (class 1)** - the ones we actually care about; precision is low (15%) and recall is 51%.
- Basically: the model struggles a lot more to correctly spot the "rare" reversed transactions than the normal ones.


In [ ]:
roc_auc = roc_auc_score(y_test, y_pred)
print("ROC-AUC Score:", roc_auc)

#### What this tells us
- **ROC-AUC = 0.52** -> this score tells us how well the model separates reversed vs not-reversed. 0.50 = pure random guessing, 1.00 = perfect.
- At 0.52, the model is only **barely better than flipping a coin.**
- This confirms what the correlation heatmap already hinted at — the data just doesn't have strong signals for this target.


In [ ]:
results_df = pd.DataFrame([{
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1,
    "ROC-AUC": roc_auc
}])
results_df

In [ ]:
print("training Accuracy:", log_model.score(x_train_scaled, y_train))
print("testing Accuracy:", log_model.score(x_test_scaled, y_test))

#### What this tells us
- Training accuracy (56%) and testing accuracy (53%) are **close to each other** - so the model isn't overfitting (it's not just memorizing the training data).
- The real problem isn't overfitting - it's that the **features we have just don't explain reversals very well**.The model is being honest, the data is just weak on signal.
